[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C11_RAG_Retrieval_Course/03_reranking/03_reranking.ipynb)

# 03 · 重排（纯 numpy/pandas）

目标：把 **bi-encoder vs cross-encoder、两段式流水线、重排提升(nDCG/MRR)、MMR 多样性去冗余、RRF 倒数排名融合、分数加权的脆弱性** 全部从零实现，并用 `assert` 验证。

路线：bi vs cross 打分 → 两段式 召回+重排 → 量化重排提升 → MMR 去冗余 → RRF 融合 → 加权 vs RRF 稳健性 → ✏️ 练习 → 📖 答案 → 🧪 真实数据(SQuAD)胶囊。

> 心智模型：**第一段召回决定『有没有把对的捞进来』，第二段重排决定『能不能把对的排到最前』**。cross-encoder 看 query-doc 交互(bi 看不到)，所以更准但更贵。

## 1 · bi-encoder vs cross-encoder：打分对比

**bi-encoder**：查询、文档各自编成向量，分数 = 余弦（只比向量，看不到词级匹配）。
**cross-encoder**：把 query-doc 一起看，分数能利用**交互特征**（如词重叠）——这正是 bi 丢掉的信息。

我们用 `cross = α·cosine + β·词重叠率` 当玩具 cross-encoder，演示它如何捕捉 bi 看不到的精确匹配。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

def stable_hash(s):
    h = 1469598103934665603
    for ch in s:
        h = (h ^ ord(ch)) * 1099511628211 % (2**64)
    return h

def embed(text, dim=64):
    '''确定性哈希词袋(玩具 bi-encoder 编码)。'''
    v = np.zeros(dim)
    for tok in text.lower().split():
        v[stable_hash(tok) % dim] += 1.0
    return v

def l2(v, eps=1e-12):
    return v / (np.linalg.norm(v) + eps)

def bi_score(q, d):
    '''bi-encoder: 两独立向量的余弦。'''
    return float(l2(embed(q)) @ l2(embed(d)))

def word_overlap(q, d):
    qs, ds = set(q.lower().split()), set(d.lower().split())
    return len(qs & ds) / (len(qs) + 1e-12)

def cross_score(q, d, alpha=0.5, beta=0.5):
    '''cross-encoder(玩具): 余弦 + 词重叠(交互特征, bi 看不到)。'''
    return alpha * bi_score(q, d) + beta * word_overlap(q, d)

q = 'machine learning model training'
docs = [
    'training a machine learning model on data',   # 高词重叠 + 高语义
    'deep neural networks and optimization',       # 语义相关但词重叠低
    'a recipe for chocolate cake',                 # 无关
]
df = pd.DataFrame({
    'doc': [d[:38] for d in docs],
    'bi(余弦)': [round(bi_score(q, d), 3) for d in docs],
    'cross(余弦+重叠)': [round(cross_score(q, d), 3) for d in docs],
})
print(df.to_string(index=False))
# cross-encoder 因看到词重叠，把『精确匹配』文档拉得更开
assert cross_score(q, docs[0]) > bi_score(q, docs[0]), 'cross 利用词重叠应给精确匹配更高分'
assert cross_score(q, docs[0]) == max(cross_score(q, d) for d in docs), 'cross 应把最匹配文档排第一'
print('\n✅ cross-encoder 利用 query-doc 交互(词重叠)，对精确匹配比 bi-encoder 更敏感')

## 2 · 两段式流水线：bi 召回 top-N → cross 重排 top-k

第一段用便宜的 bi-encoder 从全库召回 top-N（广撒网）；第二段用 cross-encoder 只对这 N 个**重排**成 top-k（精挑）。

对拍：两段式的结果应与**直接对全库跑 cross-encoder**（昂贵但精确）一致——只要相关文档进了 top-N。

In [ ]:
corpus = [
    'training a machine learning model on labeled data',
    'gradient descent optimizes model parameters',
    'machine learning requires large training datasets',
    'neural network architectures for deep learning',
    'a guide to baking sourdough bread at home',
    'the history of ancient roman architecture',
    'hyperparameter tuning for model training',
    'photosynthesis converts light into chemical energy',
]
q = 'machine learning model training'

def stage1_recall(q, corpus, N):
    '''bi-encoder 召回 top-N 下标。'''
    scores = np.array([bi_score(q, d) for d in corpus])
    return np.argsort(-scores)[:N]

def stage2_rerank(q, corpus, cand_idx, k):
    '''cross-encoder 对候选重排，返回 top-k 下标。'''
    cs = np.array([cross_score(q, corpus[i]) for i in cand_idx])
    order = np.argsort(-cs)
    return cand_idx[order][:k]

def two_stage(q, corpus, N=5, k=3):
    cand = stage1_recall(q, corpus, N)
    return stage2_rerank(q, corpus, cand, k)

def cross_full(q, corpus, k=3):
    '''昂贵参考：对全库跑 cross-encoder。'''
    cs = np.array([cross_score(q, d) for d in corpus])
    return np.argsort(-cs)[:k]

ts = two_stage(q, corpus, N=5, k=3)
ref = cross_full(q, corpus, k=3)
print('两段式 top-3 :', ts.tolist(), '->', [corpus[i][:32] for i in ts])
print('全量cross top-3:', ref.tolist())
# N 足够大时两段式 top-1 应与全量 cross 一致
assert ts[0] == ref[0], '相关文档进了候选时, 两段式 top-1 == 全量 cross top-1'
print('✅ 两段式(bi召回+cross重排) 用一小部分 cross 调用逼近全量 cross 的精度')

## 3 · 量化重排提升：nDCG / MRR

构造一个场景：真正相关的文档被 bi-encoder 排在**靠后**（信息有损），cross-encoder 能把它**提前**。
用从零实现的 **MRR** 和 **nDCG** 度量「重排后」相比「第一段」的提升。

$\text{MRR}=1/\text{rank}_{\text{第一个相关}}$，$\text{nDCG@}k=\text{DCG@}k/\text{IDCG@}k$。

In [ ]:
def dcg_at_k(rels, k):
    '''rels: 按排序位置的相关性等级列表。DCG = Σ (2^rel - 1)/log2(i+2)。'''
    rels = np.asarray(rels[:k], dtype=float)
    discounts = 1.0 / np.log2(np.arange(2, len(rels) + 2))
    return float(((2 ** rels - 1) * discounts).sum())

def ndcg_at_k(rels, k):
    '''rels: 当前排序下各位置的相关性。归一化到 [0,1]。'''
    ideal = sorted(rels, reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg_at_k(rels, k) / idcg if idcg > 0 else 0.0

def mrr(rels):
    '''第一个相关(rel>0)结果名次的倒数。'''
    for i, r in enumerate(rels):
        if r > 0:
            return 1.0 / (i + 1)
    return 0.0

# 相关性标注: 文档 6('hyperparameter tuning...') 是高相关(rel=2)，但词面与查询差，bi 排得靠后
rel_map = {0: 2, 2: 2, 6: 2, 1: 1, 3: 1, 4: 0, 5: 0, 7: 0}  # 下标->相关性等级

first_stage = stage1_recall(q, corpus, N=8)           # bi 排序(全部)
reranked = stage2_rerank(q, corpus, first_stage, k=8) # cross 重排

rels_first = [rel_map[i] for i in first_stage]
rels_rerank = [rel_map[i] for i in reranked]
print('第一段(bi)  相关性序列:', rels_first, '  nDCG@5=%.3f MRR=%.3f' % (ndcg_at_k(rels_first,5), mrr(rels_first)))
print('重排(cross) 相关性序列:', rels_rerank, '  nDCG@5=%.3f MRR=%.3f' % (ndcg_at_k(rels_rerank,5), mrr(rels_rerank)))
assert ndcg_at_k(rels_rerank, 5) >= ndcg_at_k(rels_first, 5), '重排应不降低 nDCG'
assert mrr(rels_rerank) >= mrr(rels_first), '重排应不降低 MRR'
print('✅ 重排把高相关但词面弱的文档提前 -> nDCG/MRR 提升(重排的价值有了硬证据)')

## 4 · MMR：多样性去冗余

只追求相关会让 top-k 全是 **near-duplicate**(同一信息的重复)。MMR 每步选「与查询相关 **且** 与已选不相似」的：

$\text{MMR}=\arg\max_{d_i}\,[\,\lambda\,\text{sim}(q,d_i)-(1-\lambda)\max_{d_j\in S}\text{sim}(d_i,d_j)\,]$

给一组含近重复的文档，对比纯相关性排序 vs MMR：MMR 的 top-k 信息更多样。

In [ ]:
# 候选: 0/1/2 几乎是同一句(near-duplicate), 3 是不同主题但也相关
cand_docs = [
    'the cat sat on the mat',
    'the cat sat on the mat today',     # 近重复 0
    'a cat is sitting on the mat',       # 近重复 0
    'dogs love to run in the park',      # 不同主题
]
qv = l2(embed('cat mat pets'))
dvs = np.array([l2(embed(d)) for d in cand_docs])
rel = dvs @ qv                            # 与查询的相关性

def mmr_rerank(rel, dvs, lambda_=0.5, k=3):
    '''贪心 MMR: 返回选中的下标顺序。'''
    selected, remaining = [], list(range(len(dvs)))
    while remaining and len(selected) < k:
        best, best_score = None, -1e18
        for i in remaining:
            div = max((dvs[i] @ dvs[j] for j in selected), default=0.0)  # 与已选最大相似
            score = lambda_ * rel[i] - (1 - lambda_) * div
            if score > best_score:
                best, best_score = i, score
        selected.append(best); remaining.remove(best)
    return selected

pure_rel = np.argsort(-rel)[:3].tolist()          # 纯相关性 top-3
mmr_sel = mmr_rerank(rel, dvs, lambda_=0.5, k=3)  # MMR top-3
print('纯相关性 top-3:', pure_rel, '->', [cand_docs[i][:28] for i in pure_rel])
print('MMR(λ=0.5) top-3:', mmr_sel, '->', [cand_docs[i][:28] for i in mmr_sel])

def diversity(sel, dvs):
    '''平均成对相异度(越大越多样)。'''
    if len(sel) < 2: return 0.0
    sims = [dvs[a] @ dvs[b] for i, a in enumerate(sel) for b in sel[i+1:]]
    return 1.0 - np.mean(sims)

assert 3 in mmr_sel, 'MMR 应把不同主题的文档(3)纳入 top-3 以增多样性'
assert diversity(mmr_sel, dvs) >= diversity(pure_rel, dvs), 'MMR 结果应不比纯相关性更冗余'
print('✅ MMR 把近重复挤掉、纳入不同主题文档 -> top-k 更多样, 覆盖信息面更广')

## 5 · RRF：倒数排名融合多路

多路检索(BM25 一路、稠密一路)要合并排序。RRF 只用名次：`融合分 = Σ 1/(k_const + rank)`。

在多路都靠前的文档融合分高(『多路都靠前』= 真相关的强信号)。验证融合的合理性。

In [ ]:
def reciprocal_rank_fusion(rank_lists, k_const=60):
    '''rank_lists: 多路排序(每路是文档下标降序列表)。返回按 RRF 分降序的下标列表。'''
    from collections import defaultdict
    score = defaultdict(float)
    for rl in rank_lists:
        for rank, doc in enumerate(rl):          # rank 从 0 起
            score[doc] += 1.0 / (k_const + rank + 1)
    return [d for d, _ in sorted(score.items(), key=lambda kv: -kv[1])]

# 路1(BM25): 文档2最相关; 路2(稠密): 文档2也靠前
bm25_rank  = [2, 0, 5, 1, 3]
dense_rank = [2, 1, 0, 4, 3]
fused = reciprocal_rank_fusion([bm25_rank, dense_rank])
print('BM25 排序 :', bm25_rank)
print('稠密 排序 :', dense_rank)
print('RRF 融合  :', fused)
assert fused[0] == 2, '两路都靠前的文档(2)应融合第一'
# 文档0在两路都进前三, 应高于只在一路出现的文档
assert fused.index(0) < fused.index(4), '两路都靠前的应排在只一路靠前的之前'
print('✅ RRF 只用名次即可稳健融合多路, 奖励『多路都靠前』的文档')

## 6 · 分数加权的脆弱性 vs RRF 的稳健性

分数加权依赖**归一化方式**——换一种归一化, 融合结果就抖。RRF 只用名次, **不受归一化影响**。

我们对同两路分数, 用两种归一化(min-max / z-score)做加权融合, 看 top-1 是否变化; 再看 RRF 是否纹丝不动。

In [ ]:
# 两路原始分数(量纲不同: BM25 无上界, 余弦在[-1,1])
bm25_scores  = np.array([16.2, 7.9, 3.6, 23.2, 11.9])  # 文档 0..4
dense_scores = np.array([0.67, 0.20, 0.94, 0.37, 0.11])

def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-12)
def zscore(x):
    return (x - x.mean()) / (x.std() + 1e-12)

def weighted_top1(norm):
    fused = 0.5 * norm(bm25_scores) + 0.5 * norm(dense_scores)
    return int(np.argmax(fused))

t1_minmax = weighted_top1(minmax)
t1_zscore = weighted_top1(zscore)
print(f'加权融合 top-1: min-max -> 文档{t1_minmax};  z-score -> 文档{t1_zscore}')

# RRF: 只用名次
bm25_rank  = np.argsort(-bm25_scores).tolist()
dense_rank = np.argsort(-dense_scores).tolist()
rrf_top1 = reciprocal_rank_fusion([bm25_rank, dense_rank])[0]
print(f'RRF 融合 top-1: {rrf_top1} (与归一化无关)')

# 同一组分数, 仅换归一化方式, 加权融合的 top-1 就变了 -> 脆弱!
assert t1_minmax != t1_zscore, '本例中 min-max 与 z-score 给出不同 top-1(演示脆弱性)'
# RRF 对分数的单调变换不变: 缩放分数不改变名次 -> 不改变 RRF
bm25_scaled = bm25_scores * 1000
rrf_scaled = reciprocal_rank_fusion([np.argsort(-bm25_scaled).tolist(), dense_rank])[0]
assert rrf_scaled == rrf_top1, 'RRF 对分数缩放不变(只看名次)'
print('✅ 仅换归一化, 加权融合 top-1 就从', t1_minmax, '变成', t1_zscore, '(脆弱); RRF 只看名次, 对量纲/缩放免疫(稳健)')

---
## ✏️ 练习 1：实现 cross-encoder 打分

实现 `cross_encoder_score(q, d, alpha, beta)`：结合**语义相似(余弦)**与**词级交互(词重叠率)**。
这模拟 cross-encoder 能看到、而 bi-encoder 看不到的 query-doc 交互信号。

（用上面已定义的 `bi_score` 与 `word_overlap`；词重叠率 = |q∩d 词| / |q 词|。）

In [ ]:
def cross_encoder_score(q, d, alpha=0.5, beta=0.5):
    # TODO: 返回 alpha*余弦(bi_score) + beta*词重叠率(word_overlap)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
qq = 'apple banana'
d_exact = 'apple banana cherry'      # 高词重叠
d_none  = 'orange grape melon'        # 零词重叠
s_exact = cross_encoder_score(qq, d_exact)
s_none = cross_encoder_score(qq, d_none)
assert s_exact > s_none, '词重叠高的文档 cross 分应更高'
# beta=0 时退化为纯 bi(余弦)
assert abs(cross_encoder_score(qq, d_exact, alpha=1.0, beta=0.0) - bi_score(qq, d_exact)) < 1e-9
# alpha=0,beta=1 时 = 纯词重叠率
assert abs(cross_encoder_score(qq, d_exact, alpha=0.0, beta=1.0) - word_overlap(qq, d_exact)) < 1e-9
print(f'cross(精确匹配)={s_exact:.3f} > cross(无重叠)={s_none:.3f}')
print('✅ 练习 1 通过：cross-encoder 打分正确融合语义与词级交互')

## ✏️ 练习 2：实现 MMR 重排

实现 `mmr(query_vec, doc_vecs, lambda_, k)`：贪心选 k 个文档, 每步最大化 `λ·sim(q,d) - (1-λ)·max sim(d, 已选)`, 返回选中下标的**顺序列表**。

（doc_vecs 已归一化, 用点积当相似度; 第一个选中的应是与查询最相关的。）

In [ ]:
def mmr(query_vec, doc_vecs, lambda_=0.5, k=3):
    # TODO: 贪心选 k 个: 每步对每个未选文档算 λ*sim(q,d)-(1-λ)*max_{已选} sim(d,sel)
    #       选分数最大的; 返回选中下标顺序列表
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
qv2 = l2(embed('cat mat'))
dvs2 = np.array([l2(embed(d)) for d in
    ['the cat on the mat', 'a cat on the mat too', 'dogs run in park', 'fish swim in sea']])
sel_high_lambda = mmr(qv2, dvs2, lambda_=1.0, k=2)   # 纯相关性
sel_low_lambda  = mmr(qv2, dvs2, lambda_=0.3, k=2)   # 重多样性
rel2 = dvs2 @ qv2
assert sel_high_lambda[0] == int(np.argmax(rel2)), 'λ=1 第一个应是最相关文档'
# 低 λ 时第二个选择更倾向与第一个不相似的(去冗余)
assert len(sel_low_lambda) == 2 and len(set(sel_low_lambda)) == 2, '应选出 2 个不同文档'
print('λ=1.0(纯相关) top-2:', sel_high_lambda)
print('λ=0.3(多样) top-2:', sel_low_lambda)
print('✅ 练习 2 通过：MMR 正确平衡相关性与多样性')

## ✏️ 练习 3：RRF 融合

实现 `rrf(rank_lists, k_const=60)`：输入多路排序(每路文档下标降序), 返回按 RRF 分**降序**的下标列表。

验证：两路都第一的文档融合第一; 单路时 RRF 还原原排序; 对分数缩放(名次不变)结果不变。

In [ ]:
def rrf(rank_lists, k_const=60):
    # TODO: 对每路每个文档累加 1/(k_const + rank + 1), 返回按融合分降序的下标列表
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
f = rrf([[2, 0, 1, 3], [2, 1, 0, 3]])
assert f[0] == 2 and f[-1] == 3, '两路都第一->融合第一; 都垫底->融合垫底'
assert rrf([[5, 3, 1, 0]]) == [5, 3, 1, 0], '单路 RRF 应还原原排序'
# k_const 越大, 头部名次差异被弱化(但 top-1 仍稳定)
assert rrf([[2, 0, 1, 3], [2, 1, 0, 3]], k_const=1)[0] == 2
print('RRF 融合:', f)
print('✅ 练习 3 通过：RRF 正确融合多路排序')

## ✏️ 练习 4：量化重排提升

实现 `rerank_gain(first_order, rerank_order, rel_map, k)`：给第一段排序、重排后排序、相关性标注, 返回 `(nDCG@k 提升, MRR 提升)` = 重排后指标 - 第一段指标。

（用上面已实现的 `ndcg_at_k` 和 `mrr`; rel_map 是 下标->相关性等级 的 dict。）

In [ ]:
def rerank_gain(first_order, rerank_order, rel_map, k=5):
    # TODO: 把两个排序映射成相关性序列, 分别算 nDCG@k 与 MRR, 返回 (Δndcg, Δmrr)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rmap = {0: 0, 1: 0, 2: 2, 3: 0}        # 仅文档2相关(rel=2), 其余不相关
first = [0, 1, 3, 2]                    # 唯一相关文档2 排第4
rerank = [2, 3, 0, 1]                   # 重排把2提到第1
d_ndcg, d_mrr = rerank_gain(first, rerank, rmap, k=4)
assert d_ndcg > 0, '把相关文档提前应提升 nDCG'
assert abs(d_mrr - (1.0 - 1.0/4)) < 1e-9, 'MRR 从 1/4 提到 1/1, 提升 0.75'
print(f'nDCG@4 提升 = {d_ndcg:.3f},  MRR 提升 = {d_mrr:.3f}')
print('✅ 练习 4 通过：能量化重排带来的排序质量提升')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cross_encoder_score(q, d, alpha=0.5, beta=0.5):
    return alpha * bi_score(q, d) + beta * word_overlap(q, d)

In [ ]:
# 练习 2 参考答案
def mmr(query_vec, doc_vecs, lambda_=0.5, k=3):
    rel = doc_vecs @ query_vec
    selected, remaining = [], list(range(len(doc_vecs)))
    while remaining and len(selected) < k:
        best, best_score = None, -1e18
        for i in remaining:
            div = max((doc_vecs[i] @ doc_vecs[j] for j in selected), default=0.0)
            score = lambda_ * rel[i] - (1 - lambda_) * div
            if score > best_score:
                best, best_score = i, score
        selected.append(best); remaining.remove(best)
    return selected

In [ ]:
# 练习 3 参考答案
def rrf(rank_lists, k_const=60):
    from collections import defaultdict
    score = defaultdict(float)
    for rl in rank_lists:
        for rank, doc in enumerate(rl):
            score[doc] += 1.0 / (k_const + rank + 1)
    return [d for d, _ in sorted(score.items(), key=lambda kv: -kv[1])]

In [ ]:
# 练习 4 参考答案
def rerank_gain(first_order, rerank_order, rel_map, k=5):
    rels_first = [rel_map[i] for i in first_order]
    rels_rerank = [rel_map[i] for i in rerank_order]
    d_ndcg = ndcg_at_k(rels_rerank, k) - ndcg_at_k(rels_first, k)
    d_mrr = mrr(rels_rerank) - mrr(rels_first)
    return d_ndcg, d_mrr

---
## 🧪 真实数据胶囊：在 SQuAD 上做两段式检索 + 重排

用真实 **SQuAD** 问答数据(每个问题对应一个真实段落)。优先**联网下载**, 失败则**回退到内置真实样本**。

任务：把段落当语料, 问题当查询, 比较『纯 bi-encoder 召回』vs『bi 召回 + cross 重排』把正确段落排到 top-1 的命中率。

In [ ]:
def load_squad_samples(n=6):
    '''优先联网拉真实 SQuAD; 失败回退到内置真实样本(逐字摘自 dev 集)。'''
    try:
        import urllib.request, json
        url = 'https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json'
        with urllib.request.urlopen(url, timeout=5) as f:
            data = json.load(f)
        out = []
        for art in data['data']:
            for para in art['paragraphs']:
                ctx = para['context']
                for qa in para['qas']:
                    if not qa.get('is_impossible', False) and qa['answers']:
                        out.append((qa['question'], ctx))
                        break                       # 每段落只取 1 题 -> n 个不同段落
                if len(out) >= n: break
            if len(out) >= n: break
        print(f'✅ 联网加载 SQuAD 成功, 取 {len(out)} 条(各属不同段落)')
        return out
    except Exception as e:
        print(f'⚠ 联网失败({type(e).__name__}), 回退内置真实 SQuAD 样本')
        return [
            ('In what country is Normandy located?',
             'The Normans were the people who in the 10th and 11th centuries gave their '
             'name to Normandy, a region in France.'),
            ('What is the capital of France?',
             'Paris is the capital and most populous city of France, situated on the Seine.'),
            ('What organelle performs photosynthesis?',
             'In plants, photosynthesis takes place in chloroplasts, which contain chlorophyll.'),
            ('Who developed the theory of general relativity?',
             'Albert Einstein developed the theory of general relativity, published in 1915.'),
            ('What gas do plants absorb from the atmosphere?',
             'During photosynthesis, plants absorb carbon dioxide and release oxygen.'),
            ('What is the largest planet in the solar system?',
             'Jupiter is the largest planet in the solar system, a gas giant.'),
        ]

samples = load_squad_samples(6)
questions = [q for q, _ in samples]
contexts  = [c for _, c in samples]
print(f'\n语料: {len(contexts)} 段落; 查询: {len(questions)} 问题')
print('例:', questions[0][:60])

**🧪 胶囊练习**：实现 `top1_hit_rate(questions, contexts, use_rerank)`：每个问题 `i` 的正确段落是 `contexts[i]`。
`use_rerank=False` 只用 bi-encoder 召回 top-1; `use_rerank=True` 用 bi 召回 top-N(=4) 再 cross 重排取 top-1。
返回 top-1 命中率(检索到的 top-1 下标==i 的比例)。

In [ ]:
def top1_hit_rate(questions, contexts, use_rerank=False, N=4):
    # TODO: 对每个问题 i:
    #   bi 召回全部并排序; 若 use_rerank, 取 top-N 用 cross_score 重排; 取 top-1
    #   若 top-1 下标==i 记命中; 返回命中率
    raise NotImplementedError

In [ ]:
# 自测
n_q = len(questions)
acc_bi = top1_hit_rate(questions, contexts, use_rerank=False)
acc_rr = top1_hit_rate(questions, contexts, use_rerank=True)
print(f'纯 bi 召回   top-1 命中率: {acc_bi:.2f}')
print(f'bi+cross重排 top-1 命中率: {acc_rr:.2f}')
print(f'随机基线           命中率: {1.0/n_q:.2f}')
assert 0.0 <= acc_bi <= 1.0 and 0.0 <= acc_rr <= 1.0
# 重排利用词重叠(问题与答案段落常有共享词), 命中率应不低于纯 bi 且优于随机
assert acc_rr >= acc_bi, '加重排不应降低命中率(候选含正确段落时)'
assert acc_rr > 1.0/n_q, '重排后应显著优于随机基线'
print('✅ 胶囊练习通过：真实 SQuAD 上两段式重排不劣于纯召回且优于随机')

In [ ]:
# 📖 胶囊参考答案
def top1_hit_rate(questions, contexts, use_rerank=False, N=4):
    hits = 0
    for i, q in enumerate(questions):
        bi = np.array([bi_score(q, d) for d in contexts])
        order = np.argsort(-bi)
        if use_rerank:
            cand = order[:N]
            cs = np.array([cross_score(q, contexts[j]) for j in cand])
            top1 = int(cand[np.argmax(cs)])
        else:
            top1 = int(order[0])
        hits += (top1 == i)
    return hits / len(questions)

### 小结
- **两段式**：bi-encoder/BM25 召回 top-N(广) → cross-encoder 重排 top-k(精); 一广一精, 接力赛。
- **bi vs cross**：bi 各自编码可预计算(快但无交互); cross 把 query-doc 一起看(每层交互, 准但每候选实时跑, 贵)。
- **重排提升**：把高相关但词面弱的文档提前, 提升 nDCG/MRR(位置敏感指标); **不改变召回**(候选没变)。
- **ColBERT** late interaction(token 级 MaxSim)是 bi↔cross 精度-成本谱的折中。
- **MMR** 去冗余: `λ·相关 -(1-λ)·与已选相似`, 避免 top-k 全是近重复; 先保相关再去冗余。
- **RRF** 只用名次融合多路: `Σ1/(k+rank)`, 无需归一化、对量纲免疫, 比脆弱的分数加权稳健。
- **候选集 N**: 太小漏相关(重排救不回), 太大重排贵; 看 Recall@N 曲线定 N; 端到端先定位瓶颈在哪一棒。

下一站：**模块 04 · RAG 评测** —— 答案对不对? 是没检索到、检索到噪声、还是在编? 把幻觉与漏检分开诊断。